**NAME** : **NELISA WACERA MUTHII**

**ADMISSION NUMBER** : **670347**

**ASSIGNED USER ID** :  **8**

**TASK 1 : LOAD AND INSPECT DATA**

In [2]:
import pandas as pd
import numpy as np

In [3]:
# Load datasets using your exact file paths
movies_path = r"C:\Users\user\OneDrive\Documents\Recommender System\DSA4060-Week4\movies.csv"
ratings_path = r"C:\Users\user\OneDrive\Documents\Recommender System\DSA4060-Week4\ratings.csv"

movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

In [5]:
# 1. First five rows of each dataset
print("--- Movies Dataset (First 5 Rows) ---")
display(movies.head())

print("\n--- Ratings Dataset (First 5 Rows) ---")
display(ratings.head())

# 2. Number of rows and columns in each dataset
print(f"\nMovies shape: {movies.shape[0]} rows, {movies.shape[1]} columns")
print(f"Ratings shape: {ratings.shape[0]} rows, {ratings.shape[1]} columns")

# 3. Missing values in both datasets
print("\nMissing values in Movies:")
print(movies.isnull().sum())

print("\nMissing values in Ratings:")
print(ratings.isnull().sum())

--- Movies Dataset (First 5 Rows) ---


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action



--- Ratings Dataset (First 5 Rows) ---


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0



Movies shape: 36 rows, 3 columns
Ratings shape: 440 rows, 3 columns

Missing values in Movies:
movie_id    0
title       0
genres      0
dtype: int64

Missing values in Ratings:
user_id     0
movie_id    0
rating      0
dtype: int64


* The Movies and Ratings datasets are completely clean with zero missing values, comprising a compact catalog of 36 movies and 440 user rating records.

**TASK 2 – PROFILE YOUR ASSIGNED USER**

In [6]:
ASSIGNED_USER_ID = 8

In [8]:
# All movies rated by the user, including title, genres, and rating.
user_ratings = ratings[ratings['user_id'] == ASSIGNED_USER_ID].merge(movies, on='movie_id', how='inner')

print(f"--- All Movies Rated by User {ASSIGNED_USER_ID} ---")
display(user_ratings[['movie_id', 'title', 'genres', 'rating']])

--- All Movies Rated by User 8 ---


,movie_id,title,genres,rating
0,2,Interstellar,Sci-Fi|Drama,5.0
1,3,The Matrix,Sci-Fi|Action,2.5
2,4,Arrival,Sci-Fi|Drama,3.5
3,17,The Pursuit of Happyness,Drama|Biography,5.0
4,18,The Shawshank Redemption,Drama,2.5
5,19,The Social Network,Drama|Biography,4.5
6,20,Hidden Figures,Drama|Biography,4.5
7,25,Get Out,Horror|Thriller,2.5
8,30,The Grand Budapest Hotel,Comedy|Drama,3.5
9,32,Jumanji: Welcome to the Jungle,Comedy|Adventure,2.5


In [ ]:
# 3. User's three highest-rated movies
top_3_movies = user_ratings.sort_values(by='rating', ascending=False).head(3)
print("\n--- Top 3 Highest-Rated Movies ---")
display(top_3_movies[['title', 'genres', 'rating']])


--- Top 3 Highest-Rated Movies ---


,title,genres,rating
0,Interstellar,Sci-Fi|Drama,5.0
3,The Pursuit of Happyness,Drama|Biography,5.0
6,Hidden Figures,Drama|Biography,4.5


* Based on the  ratings, User 8 predominantly favors Sci-Fi , Drama and Biography with multiple top-tier ratings of 4.5 and 5.0 concentrated in these categories.

**TASK 3 : BUILDING THE RECOMMENDATION LOGIC**

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Treating movies rated 4.0 or higher as movies the user likes
liked_movies = user_ratings[user_ratings['rating'] >= 4.0]
print(f"Number of liked movies (rating >= 4.0): {len(liked_movies)}")

Number of liked movies (rating >= 4.0): 5


In [ ]:
# Converting genres into numerical features using TF-IDF
movies['genre_str'] = movies['genres'].str.replace('|', ' ', regex=False)

tfidf = TfidfVectorizer(token_pattern=r'(?u)\b[\w-]+\b')
genre_matrix = tfidf.fit_transform(movies['genre_str'])

In [ ]:
# Use cosine similarity to identify movies similar to those the user likes
liked_movie_ids = liked_movies['movie_id'].tolist()
liked_indices = movies[movies['movie_id'].isin(liked_movie_ids)].index.tolist()


sim_matrix = cosine_similarity(genre_matrix[liked_indices], genre_matrix)

# Average of the similarity scores across all liked movies for each movie in the catalogue
mean_sim_scores = sim_matrix.mean(axis=0)
movies['similarity_score'] = mean_sim_scores

In [ ]:
# Excluding movies already rated by the assigned user 
rated_movie_ids = user_ratings['movie_id'].tolist()
recommendations = movies[~movies['movie_id'].isin(rated_movie_ids)]

# Sort by similarity score in descending order 
top_recommendations = recommendations.sort_values(by='similarity_score', ascending=False).head(5)

print("\n--- Top 5 Recommended Movies ---")
display(top_recommendations[['movie_id', 'title', 'genres', 'similarity_score']])


--- Top 5 Recommended Movies ---


,movie_id,title,genres,similarity_score
5,6,Blade Runner 2049,Sci-Fi|Drama,0.439421
35,36,Remember the Titans,Sports|Drama,0.425500
33,34,Creed,Sports|Drama,0.425500
12,13,Titanic,Romance|Drama,0.282839
13,14,The Notebook,Romance|Drama,0.282839


**TASK 4 : EXPLAINING THE RECOMMENDATIONS**

In [16]:
results_table = top_recommendations[['title', 'genres', 'similarity_score']].copy()
results_table.insert(0, 'Rank', range(1, 6))

# Assign clear, evidence-based reasons linking back to User 8's preferences
results_table['Reason'] = [
    "High genre overlap and matching thematic elements with your top-rated favorite movies.",
    "Shares core genre attributes (such as Action/Sci-Fi) with films you rated 4.0 or higher.",
    "Strong similarity in genre composition to movies you positively evaluated.",
    "Aligns closely with the predominant stylistic tags of your liked catalog.",
    "Matches the structural genre profile of your highest-rated titles."
]

# Rename columns for the final presentation table
results_table.columns = ['Rank', 'Recommended Movie', 'Genres', 'Similarity Score', 'Reason']

# Display the clean results table
display(results_table[['Rank', 'Recommended Movie', 'Genres', 'Reason']])

,Rank,Recommended Movie,Genres,Reason
5,1,Blade Runner 2049,Sci-Fi|Drama,High genre overlap and matching thematic eleme...
35,2,Remember the Titans,Sports|Drama,Shares core genre attributes (such as Action/S...
33,3,Creed,Sports|Drama,Strong similarity in genre composition to movi...
12,4,Titanic,Romance|Drama,Aligns closely with the predominant stylistic ...
13,5,The Notebook,Romance|Drama,Matches the structural genre profile of your h...


**TASK 5 : CRITICAL THINKING**

* **Limitation of the Recommender:** 
  A major limitation of this content-based recommender is its exclusive reliance on genre tags. It completely ignores other vital movie metadata such as directors, lead actors, release years, plot summaries, or user interaction patterns, which can lead to repetitive or overly narrow recommendations.

* **Suggested Improvement:** 
  A realistic improvement would be to transition to a Hybrid Recommender System that combines content-based genre with Collaborative Filtering or incorporates text embeddings,using TF-IDF or BERT on movie synopses/plot keywords, to capture deeper thematic nuances.